# Experiment 3: Logging tokens, latency and cost to MongoDB; timeouts and retries
**Course:** 25CS2207E Generative AI and AI Platform Engineering · **Module I** · CO1

**Aim:** Log every call as one MongoDB document and make the client resilient: timeouts, retries with exponential backoff for 429/5xx, fail-fast for 4xx.

**Before you run:** add `GEMINI_API_KEY`; for MongoDB, add `MONGODB_URI` from a free Atlas cluster with Network Access 0.0.0.0/0. Without it the log goes to `results/calls.jsonl`.

Run the cells top to bottom (Runtime → Run all). Copy the printed numbers into the *Observations* section of your lab record.

In [ ]:
!pip -q install google-genai python-dotenv requests pymongo

In [1]:
# Copy Colab Secrets (key icon in the left sidebar) into environment variables.
# Add GEMINI_API_KEY (required) and MONGODB_URI (optional), and turn on "Notebook access".
import os
try:
    from google.colab import userdata
    for name in ["GEMINI_API_KEY", "MONGODB_URI"]:
        try:
            os.environ[name] = userdata.get(name)
            print(f"{name}: loaded")
        except Exception:
            print(f"{name}: not set" + (" (logs will go to results/calls.jsonl)" if name == "MONGODB_URI" else " - add it in Secrets!"))
except ImportError:
    print("Not running in Colab: set the variables in a .env file instead.")
os.environ.setdefault("GEMINI_MODEL", "gemini-3.5-flash-lite")

GEMINI_API_KEY: loaded
MONGODB_URI: loaded


'gemini-3.5-flash-lite'

In [2]:
!mkdir -p gale config data results experiments

In [3]:
%%writefile gale/__init__.py
"""GALE course assistant - Module I code (Experiments 1-6)."""

Writing gale/__init__.py


In [4]:
%%writefile gale/config.py
"""12-factor configuration: every setting comes from environment variables.

In VS Code the values are read from a local `.env` file (never committed).
In Google Colab the notebooks copy Colab Secrets into the environment first.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from pathlib import Path

try:  # python-dotenv is optional so the code still runs without it
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:  # pragma: no cover
    pass

ROOT = Path(__file__).resolve().parent.parent
RESULTS_DIR = Path(os.getenv("GALE_RESULTS_DIR", ROOT / "results"))
DATA_DIR = Path(os.getenv("GALE_DATA_DIR", ROOT / "data"))
CONFIG_DIR = Path(os.getenv("GALE_CONFIG_DIR", ROOT / "config"))


def _env_float(name: str, default: float | None) -> float | None:
    raw = os.getenv(name)
    return float(raw) if raw not in (None, "") else default


def _env_int(name: str, default: int) -> int:
    raw = os.getenv(name)
    return int(raw) if raw not in (None, "") else default


@dataclass
class Settings:
    provider: str = field(default_factory=lambda: os.getenv("LLM_PROVIDER", "gemini"))
    gemini_api_key: str | None = field(default_factory=lambda: os.getenv("GEMINI_API_KEY"))
    gemini_model: str = field(default_factory=lambda: os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"))
    ollama_host: str = field(default_factory=lambda: os.getenv("OLLAMA_HOST", "http://localhost:11434"))
    ollama_model: str = field(default_factory=lambda: os.getenv("OLLAMA_MODEL", "qwen2.5:0.5b"))
    mongodb_uri: str | None = field(default_factory=lambda: os.getenv("MONGODB_URI"))
    mongodb_db: str = field(default_factory=lambda: os.getenv("MONGODB_DB", "gale"))
    timeout_s: float = field(default_factory=lambda: _env_float("LLM_TIMEOUT_S", 30.0))
    max_retries: int = field(default_factory=lambda: _env_int("LLM_MAX_RETRIES", 3))
    request_delay_s: float = field(default_factory=lambda: _env_float("REQUEST_DELAY_S", 4.0))
    # None = use the provider's default temperature (Google recommends the default for Gemini 3.x)
    temperature: float | None = field(default_factory=lambda: _env_float("LLM_TEMPERATURE", None))
    # Thinking tokens share this budget on Gemini, so keep it generous
    max_output_tokens: int = field(default_factory=lambda: _env_int("LLM_MAX_OUTPUT_TOKENS", 1024))


def get_settings() -> Settings:
    return Settings()


def load_pricing() -> dict:
    """Price table in USD per 1M tokens, keyed by model id."""
    with open(CONFIG_DIR / "pricing.json", encoding="utf-8") as f:
        return json.load(f)


def ensure_results_dir() -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    return RESULTS_DIR

Writing gale/config.py


In [5]:
%%writefile config/pricing.json
{
  "_note": "Paid Standard tier, USD per 1M tokens, from https://ai.google.dev/gemini-api/docs/pricing read on 2026-11-05. Output price includes thinking tokens. Flash prices double from 2027-01-01; update before use. Free tier is $0 but rate-limited.",
  "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
  "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
  "gemini-3.6-flash": {"input": 0.75, "output": 3.75},
  "gemini-3.8-flash": {"input": 0.75, "output": 3.75},
  "qwen2.5:0.5b": {"input": 0.0, "output": 0.0},
  "llama3.2:1b": {"input": 0.0, "output": 0.0},
  "fake-model": {"input": 0.0, "output": 0.0}
}

Writing config/pricing.json


In [6]:
%%writefile gale/llm_client.py
"""Provider-agnostic LLM client (Experiments 1-3).

One interface (`LLMClient.generate`) over several back ends:
  * GeminiProvider  - hosted API (Google Gemini, free tier)
  * OllamaProvider  - local open-weight model served by Ollama
  * FakeProvider    - deterministic offline model for unit tests
  * FlakyProvider   - wraps another provider and fails on purpose (retry demo)

Every call returns an `LLMResponse` carrying text, token counts, latency,
cost and the number of attempts, and is handed to a logger (MongoDB/JSONL).
"""
from __future__ import annotations

import random
import time
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Callable, Optional, Protocol

from .config import Settings, get_settings, load_pricing

RETRYABLE_STATUS = {408, 429, 500, 502, 503, 504}


# --------------------------------------------------------------------------- #
# Data types
# --------------------------------------------------------------------------- #
@dataclass
class RawResult:
    """What a provider returns before the client adds timing and cost."""

    text: str
    input_tokens: int
    output_tokens: int
    thinking_tokens: int = 0


@dataclass
class LLMResponse:
    request_id: str
    timestamp: str
    provider: str
    model: str
    prompt: str
    system: Optional[str]
    text: str
    input_tokens: int
    output_tokens: int
    thinking_tokens: int
    latency_ms: float
    cost_usd: float
    attempts: int
    ok: bool
    error: Optional[str] = None
    tags: dict = field(default_factory=dict)

    @property
    def total_tokens(self) -> int:
        return self.input_tokens + self.output_tokens + self.thinking_tokens

    def to_dict(self) -> dict:
        d = asdict(self)
        d["total_tokens"] = self.total_tokens
        return d


class ProviderError(Exception):
    def __init__(self, message: str, status: Optional[int] = None, retryable: bool = False):
        super().__init__(message)
        self.status = status
        self.retryable = retryable


class Provider(Protocol):
    name: str
    model: str

    def generate(self, prompt: str, system: Optional[str], temperature: float,
                 max_output_tokens: int, timeout_s: float) -> RawResult: ...


# --------------------------------------------------------------------------- #
# Providers
# --------------------------------------------------------------------------- #
class GeminiProvider:
    """Google Gemini through the official `google-genai` SDK."""

    name = "gemini"

    def __init__(self, api_key: str, model: str):
        if not api_key:
            raise ValueError("GEMINI_API_KEY is not set (see .env.example or Colab Secrets).")
        from google import genai  # imported lazily so offline tests need no SDK

        self._genai = genai
        self._client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt, system, temperature, max_output_tokens, timeout_s) -> RawResult:
        from google.genai import types

        config = types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            http_options=types.HttpOptions(timeout=int(timeout_s * 1000)),  # milliseconds
        )
        try:
            resp = self._client.models.generate_content(model=self.model, contents=prompt, config=config)
        except Exception as exc:  # map SDK/network errors to ProviderError
            raise _classify(exc) from exc
        usage = resp.usage_metadata
        return RawResult(
            text=resp.text or "",
            input_tokens=getattr(usage, "prompt_token_count", 0) or 0,
            output_tokens=getattr(usage, "candidates_token_count", 0) or 0,
            thinking_tokens=getattr(usage, "thoughts_token_count", 0) or 0,
        )


class OllamaProvider:
    """Local open-weight model served by Ollama's REST API."""

    name = "ollama"

    def __init__(self, host: str, model: str):
        self.host = host.rstrip("/")
        self.model = model

    def generate(self, prompt, system, temperature, max_output_tokens, timeout_s) -> RawResult:
        import requests

        messages = ([{"role": "system", "content": system}] if system else []) + [
            {"role": "user", "content": prompt}
        ]
        body = {
            "model": self.model,
            "messages": messages,
            "stream": False,
            "options": {"num_predict": max_output_tokens},
        }
        if temperature is not None:
            body["options"]["temperature"] = temperature
        try:
            r = requests.post(f"{self.host}/api/chat", json=body, timeout=timeout_s)
        except requests.exceptions.RequestException as exc:
            raise ProviderError(f"Ollama unreachable: {exc}", retryable=True) from exc
        if r.status_code != 200:
            raise ProviderError(f"Ollama HTTP {r.status_code}: {r.text[:200]}", r.status_code,
                                r.status_code in RETRYABLE_STATUS)
        data = r.json()
        return RawResult(
            text=data.get("message", {}).get("content", ""),
            input_tokens=data.get("prompt_eval_count", 0) or 0,
            output_tokens=data.get("eval_count", 0) or 0,
        )


class FakeProvider:
    """Deterministic offline model: answers with a canned reply, counts words as tokens."""

    name = "fake"

    def __init__(self, model: str = "fake-model", reply: Optional[Callable[[str], str]] = None,
                 delay_s: float = 0.0):
        self.model = model
        self._reply = reply or (lambda p: f"Echo: {p}")
        self.delay_s = delay_s

    def generate(self, prompt, system, temperature, max_output_tokens, timeout_s) -> RawResult:
        if self.delay_s:
            time.sleep(self.delay_s)
        text = self._reply(prompt)
        n_in = len(((system or "") + " " + prompt).split())
        return RawResult(text=text, input_tokens=n_in, output_tokens=len(text.split()))


class FlakyProvider:
    """Fails the first `fail_times` calls with `status`, then delegates. Used to test retries."""

    def __init__(self, inner: Provider, fail_times: int = 2, status: int = 503):
        self.inner = inner
        self.name = f"flaky-{inner.name}"
        self.model = inner.model
        self.fail_times = fail_times
        self.status = status
        self.calls = 0

    def generate(self, *args, **kwargs) -> RawResult:
        self.calls += 1
        if self.calls <= self.fail_times:
            raise ProviderError(f"simulated HTTP {self.status}", self.status,
                                self.status in RETRYABLE_STATUS)
        return self.inner.generate(*args, **kwargs)


def _classify(exc: Exception) -> ProviderError:
    """Turn any SDK/network exception into a ProviderError with a retryable flag."""
    status = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    try:
        status = int(status) if status is not None else None
    except (TypeError, ValueError):
        status = None
    name = type(exc).__name__.lower()
    retryable = (status in RETRYABLE_STATUS) or "timeout" in name or "connect" in name
    return ProviderError(f"{type(exc).__name__}: {exc}", status, retryable)


# --------------------------------------------------------------------------- #
# Cost
# --------------------------------------------------------------------------- #
def compute_cost(model: str, input_tokens: int, output_tokens: int, pricing: dict) -> float:
    """USD cost of one call. `output_tokens` must already include thinking tokens."""
    price = pricing.get(model)
    if price is None:
        return 0.0
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000


# --------------------------------------------------------------------------- #
# Client
# --------------------------------------------------------------------------- #
class LLMClient:
    def __init__(self, provider: Provider, logger=None, pricing: Optional[dict] = None,
                 max_retries: int = 3, timeout_s: float = 30.0, temperature: Optional[float] = None,
                 max_output_tokens: int = 1024, backoff_base_s: float = 1.0,
                 sleep: Callable[[float], None] = time.sleep):
        self.provider = provider
        self.logger = logger
        self.pricing = pricing if pricing is not None else load_pricing()
        self.max_retries = max_retries
        self.timeout_s = timeout_s
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens
        self.backoff_base_s = backoff_base_s
        self._sleep = sleep

    def generate(self, prompt: str, system: Optional[str] = None, temperature: Optional[float] = None,
                 max_output_tokens: Optional[int] = None, tags: Optional[dict] = None) -> LLMResponse:
        temperature = self.temperature if temperature is None else temperature
        max_output_tokens = self.max_output_tokens if max_output_tokens is None else max_output_tokens
        attempts, last_err, raw = 0, None, None
        start = time.perf_counter()
        while attempts <= self.max_retries:
            attempts += 1
            try:
                raw = self.provider.generate(prompt, system, temperature, max_output_tokens, self.timeout_s)
                break
            except ProviderError as err:
                last_err = err
                if not err.retryable or attempts > self.max_retries:
                    break
                # exponential backoff with jitter: 1s, 2s, 4s ... (+ up to 0.5 s)
                self._sleep(self.backoff_base_s * (2 ** (attempts - 1)) + random.uniform(0, 0.5))
        latency_ms = (time.perf_counter() - start) * 1000

        if raw is not None:
            billed_out = raw.output_tokens + raw.thinking_tokens
            resp = LLMResponse(
                request_id=str(uuid.uuid4()), timestamp=datetime.now(timezone.utc).isoformat(),
                provider=self.provider.name, model=self.provider.model, prompt=prompt, system=system,
                text=raw.text, input_tokens=raw.input_tokens, output_tokens=raw.output_tokens,
                thinking_tokens=raw.thinking_tokens, latency_ms=round(latency_ms, 1),
                cost_usd=compute_cost(self.provider.model, raw.input_tokens, billed_out, self.pricing),
                attempts=attempts, ok=True, tags=tags or {},
            )
        else:  # safe failure: never crash the caller, return an explicit error response
            resp = LLMResponse(
                request_id=str(uuid.uuid4()), timestamp=datetime.now(timezone.utc).isoformat(),
                provider=self.provider.name, model=self.provider.model, prompt=prompt, system=system,
                text="", input_tokens=0, output_tokens=0, thinking_tokens=0,
                latency_ms=round(latency_ms, 1), cost_usd=0.0, attempts=attempts, ok=False,
                error=str(last_err), tags=tags or {},
            )
        if self.logger is not None:
            self.logger.log(resp)
        return resp


def make_provider(settings: Optional[Settings] = None, name: Optional[str] = None) -> Provider:
    s = settings or get_settings()
    name = (name or s.provider).lower()
    if name == "gemini":
        return GeminiProvider(s.gemini_api_key, s.gemini_model)
    if name == "ollama":
        return OllamaProvider(s.ollama_host, s.ollama_model)
    if name == "fake":
        return FakeProvider()
    raise ValueError(f"Unknown provider '{name}'. Use gemini, ollama or fake.")


def make_client(settings: Optional[Settings] = None, provider_name: Optional[str] = None,
                logger=None) -> LLMClient:
    s = settings or get_settings()
    return LLMClient(make_provider(s, provider_name), logger=logger, max_retries=s.max_retries,
                     timeout_s=s.timeout_s, temperature=s.temperature,
                     max_output_tokens=s.max_output_tokens)

Writing gale/llm_client.py


In [7]:
%%writefile gale/call_logger.py
"""Call logging (Experiment 3).

Every LLM call becomes one document:
{
  request_id, timestamp, provider, model, prompt, system, text,
  input_tokens, output_tokens, thinking_tokens, total_tokens,
  latency_ms, cost_usd, attempts, ok, error, tags{experiment, run_id, ...}
}
MongoLogger writes to MongoDB (Atlas or local). JsonlLogger writes the same
documents to results/calls.jsonl, so the experiments still run without MongoDB.
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

from .config import RESULTS_DIR, get_settings


class JsonlLogger:
    kind = "jsonl"

    def __init__(self, path: Optional[Path] = None):
        self.path = Path(path or RESULTS_DIR / "calls.jsonl")
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, response) -> None:
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(response.to_dict(), ensure_ascii=False) + "\n")

    def all(self) -> list[dict]:
        if not self.path.exists():
            return []
        with open(self.path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


class MongoLogger:
    kind = "mongodb"

    def __init__(self, uri: str, db: str = "gale", collection: str = "llm_calls"):
        from pymongo import ASCENDING, MongoClient

        self._client = MongoClient(uri, serverSelectionTimeoutMS=5000)
        self._client.admin.command("ping")  # fail fast if the cluster is unreachable
        self.col = self._client[db][collection]
        self.col.create_index([("timestamp", ASCENDING)])
        self.col.create_index([("tags.run_id", ASCENDING)])

    def log(self, response) -> None:
        self.col.insert_one(response.to_dict())

    def all(self, query: Optional[dict] = None) -> list[dict]:
        return list(self.col.find(query or {}, {"_id": 0}))


def make_logger(prefer_mongo: bool = True):
    """MongoDB if MONGODB_URI is set and reachable, otherwise the JSONL file."""
    s = get_settings()
    if prefer_mongo and s.mongodb_uri:
        try:
            logger = MongoLogger(s.mongodb_uri, s.mongodb_db)
            print(f"[logger] MongoDB connected: db={s.mongodb_db}, collection=llm_calls")
            return logger
        except Exception as exc:  # noqa: BLE001
            print(f"[logger] MongoDB unavailable ({type(exc).__name__}); using results/calls.jsonl")
    else:
        print("[logger] MONGODB_URI not set; using results/calls.jsonl")
    return JsonlLogger()

Writing gale/call_logger.py


## Step 1 — Five logged calls

In [8]:
from gale.call_logger import make_logger
from gale.config import get_settings
from gale.llm_client import FakeProvider, FlakyProvider, LLMClient, make_provider

s = get_settings()
logger = make_logger()
client = LLMClient(make_provider(s, "gemini"), logger=logger, max_retries=3, timeout_s=30)
for p in ["What is a token?", "What is a context window?", "Why can an LLM hallucinate?",
          "What is a rate limit?", "Give one reason to log every LLM call."]:
    r = client.generate(p, system="Answer in one sentence.", tags={"experiment": "exp03"})
    print(f"{r.latency_ms:7.0f} ms in={r.input_tokens} out={r.output_tokens} think={r.thinking_tokens} "
          f"${r.cost_usd:.6f} attempts={r.attempts} ok={r.ok}")

[logger] MongoDB unavailable (ModuleNotFoundError); using results/calls.jsonl
   1104 ms in=12 out=38 think=0 $0.000099 attempts=1 ok=True
    542 ms in=13 out=32 think=0 $0.000084 attempts=1 ok=True
    527 ms in=15 out=29 think=0 $0.000077 attempts=1 ok=True
    536 ms in=13 out=26 think=0 $0.000069 attempts=1 ok=True
    611 ms in=17 out=19 think=0 $0.000053 attempts=1 ok=True


## Step 2 — Retry demo: HTTP 503 twice, then success
Backoff waits 0.5 s, then 1 s (plus jitter).

In [9]:
flaky = LLMClient(FlakyProvider(FakeProvider(), fail_times=2, status=503), logger=logger,
                  max_retries=3, backoff_base_s=0.5, pricing={"fake-model": {"input": 0, "output": 0}})
rb = flaky.generate("retry test", tags={"experiment": "exp03", "case": "retry_503"})
print(f"ok={rb.ok} attempts={rb.attempts} latency={rb.latency_ms:.0f} ms")

ok=True attempts=3 latency=2010 ms


## Step 3 — HTTP 400 is not retried (fail fast, safe error)

In [10]:
bad = LLMClient(FlakyProvider(FakeProvider(), fail_times=5, status=400), logger=logger,
                max_retries=3, backoff_base_s=0.5, pricing={"fake-model": {"input": 0, "output": 0}})
rc = bad.generate("bad request test", tags={"experiment": "exp03", "case": "error_400"})
print(f"ok={rc.ok} attempts={rc.attempts} error={rc.error}")

ok=False attempts=1 error=simulated HTTP 400


## Step 4 — Inspect the call log

In [13]:
import json, pandas as pd
docs = [d for d in logger.all() if d.get("tags", {}).get("experiment") == "exp03"]
df = pd.DataFrame(docs)
display(df[["timestamp", "provider", "model", "input_tokens", "output_tokens", "thinking_tokens",
            "latency_ms", "cost_usd", "attempts", "ok"]])
with open("results/exp03_sample_call_log.json", "w", encoding="utf-8") as f:
    json.dump(docs[-3:], f, indent=2, ensure_ascii=False, default=str)
print("Logger:", logger.kind, "| documents:", len(docs))

,timestamp,provider,model,input_tokens,output_tokens,thinking_tokens,latency_ms,cost_usd,attempts,ok
0,2026-11-13T08:13:26.776221+00:00,gemini,gemini-3.5-flash-lite,12,38,0,1104.0,0.000099,1,True
1,2026-11-13T08:13:27.318759+00:00,gemini,gemini-3.5-flash-lite,13,32,0,541.9,0.000084,1,True
2,2026-11-13T08:13:27.845854+00:00,gemini,gemini-3.5-flash-lite,15,29,0,526.7,0.000077,1,True
3,2026-11-13T08:13:28.382231+00:00,gemini,gemini-3.5-flash-lite,13,26,0,536.0,0.000069,1,True
4,2026-11-13T08:13:28.993422+00:00,gemini,gemini-3.5-flash-lite,17,19,0,610.7,0.000053,1,True
5,2026-11-13T08:13:34.217581+00:00,flaky-fake,fake-model,2,3,0,2009.6,0.000000,3,True
6,2026-11-13T08:13:43.932147+00:00,flaky-fake,fake-model,0,0,0,0.0,0.000000,1,False


Logger: jsonl | documents: 7


In [12]:
# Download every result file of this experiment for your lab record.
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>